# Figure 1: Population Descriptive Table
Run to create the population descriptive table

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import html

# --- Configuration ---
# Define the path to your input data file.
INPUT_CSV_PATH = '/home/server/Projects/data/AKI/tabular_combined_unnormalized.csv'
# Define the path for the output HTML file.
OUTPUT_HTML_PATH = 'descriptive_table.html'

def generate_descriptive_table_html(df: pd.DataFrame) -> str:
    """
    Generates a formatted descriptive statistics table as an HTML string.

    The output is a self-contained HTML file that can be opened in a browser,
    making it easy to copy and paste into rich text editors like Google Docs.

    Args:
        df: A pre-processed DataFrame with one row per subject.

    Returns:
        A string containing the full HTML of the descriptive statistics table.
    """
    # This list will hold the HTML `<tr>` strings for our table.
    table_rows_html = []
    total_population = len(df)

    # --- Helper function for formatting ---
    def add_row(characteristic: str, finding: str, is_section_header: bool = False, is_indented: bool = False):
        """Adds a formatted row to the HTML table list."""
        if is_section_header:
            # Section headers span both columns and have a border
            row_html = f'<tr><td colspan="2" style="font-weight: bold; padding-top: 10px; border: 1px solid black;">{characteristic}</td></tr>'
        else:
            # Regular rows have borders on each cell
            char_style = 'padding-left: 25px;' if is_indented else ''
            row_html = f'<tr><td style="{char_style} border: 1px solid black;">{characteristic}</td><td style="border: 1px solid black;">{finding}</td></tr>'
        table_rows_html.append(row_html)

    # --- Population Demographics ---
    continuous_vars = {
        'Age, y': 'age',
        'Weight, kg': 'weight',
        'Height, cm': 'height',
        'BMI, kg/m^2': 'BMI',
        'ASA': 'asa',
        'Number of Preexisting Cardiac Diagnoses': 'num_card_events',
        'Booking Case Length, min': 'booking_case_length'
    }
    for desc, col in continuous_vars.items():
        if col in df.columns:
            mean = df[col].mean()
            std = df[col].std()
            add_row(f'{desc}, mean (SD)', f'{mean:.2f} ± {std:.2f}')

    if 'sex' in df.columns:
        female_count = df['sex'].value_counts().get(False, 0)
        percentage = (female_count / total_population) * 100
        add_row('Female sex, n (%)', f'{female_count} ({percentage:.2f}%)')
    
    # --- ASA Classification ---
    if 'asa' in df.columns:
        add_row('ASA classification, n (%)', '', is_section_header=True)
        asa_counts = df['asa'].value_counts().sort_index()
        for level, count in asa_counts.items():
            percentage = (count / total_population) * 100
            add_row(f'{level}', f'{count} ({percentage:.3f}%)', is_indented=True)

    # --- Postoperative AKI ---
    if 'aki_boolean' in df.columns:
        aki_count = df['aki_boolean'].value_counts().get(True, 0)
        percentage = (aki_count / total_population) * 100
        add_row('Postoperative AKI, n (%)', f'{aki_count} ({percentage:.3f}%)')

    # --- Department Surgery Type ---
    # Dictionary to map abbreviations to full department names
    dept_mapping = {
        'UR': 'Urology',
        'RO': 'Radiation Oncology',
        'RAD': 'Radiology',
        'PS': 'Plastic Surgery',
        'PED': 'Pediatric Surgery',
        'OT': 'Orthopedic Surgery',
        'OS': 'Oral Surgery',
        'OL': 'Otorhinolaryngology',
        'OG': 'Obstetrics and Gynecology',
        'NS': 'Neurosurgery',
        'IM': 'Internal Medicine',
        'GS': 'General Surgery',
        'EM': 'Emergency Medicine',
        'DM': 'Dermatology',
        'CTS': 'Cardiothoracic Surgery',
        'AN': 'Anesthesiology'
    }
    
    dept_cols = sorted([col for col in df.columns if 'department' in col])
    if dept_cols:
        add_row('Department Surgery type, n (%)', '', is_section_header=True)
        for col in dept_cols:
            # Extract abbreviation from column name (e.g., 'department_GS' -> 'GS')
            abbreviation = col.replace('department_', '').upper()
            # Look up the full name, defaulting to the abbreviation if not found
            dept_name = dept_mapping.get(abbreviation, abbreviation)
            
            count = df[col].sum()
            percentage = (count / total_population) * 100
            add_row(f'{dept_name}', f'{count} ({percentage:.4f}%)', is_indented=True)

    # --- Assemble the final HTML string ---
    all_rows_str = "\n".join(table_rows_html)
    html_template = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Descriptive Statistics</title>
        <style>
            body {{ font-family: 'Times New Roman', Times, serif; margin: 20px; }}
            table {{ border-collapse: collapse; width: 600px; border: 1px solid black; }}
            th, td {{ padding: 8px 12px; text-align: left; border: 1px solid black; }}
            th {{ background-color: #f2f2f2; font-weight: bold; }}
            .title {{ font-size: 1.2em; font-weight: bold; margin-bottom: 10px; }}
        </style>
    </head>
    <body>
        <div class="title">Table 1. Characteristics of Cohort</div>
        <table>
            <thead>
                <tr>
                    <th>Characteristic</th>
                    <th>Finding (N={total_population})</th>
                </tr>
            </thead>
            <tbody>
                {all_rows_str}
            </tbody>
        </table>
    </body>
    </html>
    """
    return html_template

def main():
    """
    Main function to execute the data processing and table generation.
    """
    print("Starting script...")
    try:
        df = pd.read_csv(INPUT_CSV_PATH)
        
        # Pre-processing: one row per subject, not operation
        df_subject = df.groupby('subject_id').agg('first').reset_index()
        df_subject = df_subject.drop(columns=['op_id'], errors='ignore')

        print(f"Loaded and processed {len(df_subject)} unique subjects.")

        # Generate the descriptive table as an HTML string
        descriptive_table_html = generate_descriptive_table_html(df_subject)

        # --- Create the combined output for Jupyter Notebook ---
        # Escape the HTML for displaying it as code
        escaped_html = html.escape(descriptive_table_html)

        # Create the collapsible viewer for the raw HTML
        notebook_display_html = f"""
        {descriptive_table_html}
        <details style="margin-top: 20px; border: 1px solid #ccc; padding: 10px; border-radius: 5px;">
            <summary style="cursor: pointer; font-weight: bold; color: #0073e6;">Click to view/hide raw HTML code</summary>
            <pre style="background-color: #f8f8f8; padding: 10px; border: 1px solid #ddd; border-radius: 5px; white-space: pre-wrap; word-wrap: break-word;"><code>{escaped_html}</code></pre>
        </details>
        """

        # --- Display combined output in Jupyter Notebook ---
        print("\nDisplaying interactive table in notebook output:")
        display(HTML(notebook_display_html))

        # --- Save the clean HTML to a file ---
        with open(OUTPUT_HTML_PATH, 'w', encoding='utf-8') as f:
            f.write(descriptive_table_html)

        print("\n" + "="*50)
        print("SUCCESS!")
        print(f"A clean version of the rich text table has also been saved to '{OUTPUT_HTML_PATH}'")
        print("="*50 + "\n")

    except FileNotFoundError:
        print(f"Error: Input file not found at '{INPUT_CSV_PATH}'")
    except ImportError:
        print("\nNOTE: To display the table in the notebook, you must run this script in a Jupyter environment.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == '__main__':
    main()


Starting script...
Loaded and processed 49198 unique subjects.

Displaying interactive table in notebook output:



SUCCESS!
A clean version of the rich text table has also been saved to 'descriptive_table.html'



# Supplemental Table: Fill Rate
Creates the Fill rate Table for the supplemental

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import html

# --- Configuration ---
# Define the path to your input data file.
INPUT_CSV_PATH = '/home/server/Projects/data/AKI/tabular_combined_unnormalized.csv'
# Define the path for the output HTML file.
OUTPUT_HTML_PATH = 'fill_rate_table.html'

def calculate_and_consolidate_fill_rates(df: pd.DataFrame) -> pd.Series:
    """
    Calculates the fill rate of columns and consolidates rates for high-frequency variables.

    Args:
        df: The input DataFrame.

    Returns:
        A pandas Series with the consolidated fill rates, sorted in descending order.
    """
    # Isolate isna columns and calculate the initial fill rate
    df_isna = df.filter(like='_isna')
    if df_isna.empty:
        print("Warning: No columns ending in '_isna' found. Fill rates cannot be calculated.")
        return pd.Series(dtype=float)
        
    col_fill_rate = 1 - (df_isna.sum(axis=0) / df_isna.shape[0])
    col_fill_rate.index = col_fill_rate.index.str.replace('_isna', '', regex=False)

    # Define groups of high-frequency labels
    high_frequency_labels = [
        "rr", "hr", "spo2", "fio2", "pmean", "etco2", "peep", "pip",
        "art_mbp", "cpat", "vt", "art_sbp", "art_dbp", "minvol",
        "pplat", "bt", "etgas", "cvp"
    ]
    medium_frequency_labels = [
        "pap_mbp", "pap_sbp", "pap_dbp", "nibp_mbp", "nibp_dbp", "nibp_sbp"
    ]
    regular_labels = high_frequency_labels + medium_frequency_labels

    # Consolidate fill rates for regular labels
    for label in regular_labels:
        # Find all columns that contain the label (e.g., 'hr', 'mean_hr', etc.)
        cols_to_consolidate = col_fill_rate[col_fill_rate.index.str.contains(label)].index
        if not cols_to_consolidate.empty:
            min_fill_value = col_fill_rate[cols_to_consolidate].min()
            # Drop the individual columns
            col_fill_rate = col_fill_rate.drop(cols_to_consolidate)
            # Add the consolidated label back with the minimum fill rate
            col_fill_rate[label] = min_fill_value

    # Clean up remaining labels by removing common prefixes
    col_fill_rate.index = col_fill_rate.index.str.replace(r"^(mean_|sum_|max_|min_)", "", regex=True)
    
    # Remove duplicates that might arise after cleaning prefixes, keeping the highest fill rate
    col_fill_rate = col_fill_rate.groupby(col_fill_rate.index).max()

    return col_fill_rate.sort_values(ascending=False)

def generate_fill_rate_table_html(fill_rates: pd.Series) -> str:
    """
    Generates a formatted HTML table for the given fill rate Series.

    Args:
        fill_rates: A pandas Series containing variables and their fill rates.

    Returns:
        A string containing the full HTML of the fill rate table.
    """
    table_rows_html = []
    for variable, rate in fill_rates.items():
        # Format rate as a percentage with 2 decimal places
        rate_percent = f"{rate:.2%}"
        # Create a simple table row with two cells
        row_html = f'<tr><td>{variable}</td><td>{rate_percent}</td></tr>'
        table_rows_html.append(row_html)
    
    all_rows_str = "\n".join(table_rows_html)
    # --- HTML Template with Google Docs friendly styles ---
    html_template = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Variable Fill Rates</title>
        <style>
            body {{ 
                font-family: 'Times New Roman', Times, serif; 
                margin: 20px; 
                line-height: 1.15;
            }}
            table {{ 
                border-collapse: collapse; 
                width: 600px; 
                border: 1px solid black; 
            }}
            th, td {{ 
                padding: 8px 12px; 
                text-align: left; 
                border: 1px solid black; 
            }}
            th {{ 
                background-color: #f2f2f2; 
                font-weight: bold;
            }}
            .title {{ 
                font-size: 1.2em; 
                font-weight: bold; 
                margin-bottom: 10px; 
            }}
        </style>
    </head>
    <body>
        <div class="title">Table 2. Variable Fill Rates</div>
        <table>
            <thead>
                <tr>
                    <th>Variable</th>
                    <th>Fill Rate (%)</th>
                </tr>
            </thead>
            <tbody>
                {all_rows_str}
            </tbody>
        </table>
    </body>
    </html>
    """
    return html_template


def main():
    """
    Main function to execute the fill rate calculation and table generation.
    """
    print("Starting script...")
    try:
        df = pd.read_csv(INPUT_CSV_PATH)
        
        print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")

        # Calculate and consolidate fill rates
        fill_rates = calculate_and_consolidate_fill_rates(df)

        # Generate the descriptive table as an HTML string
        fill_rate_table_html = generate_fill_rate_table_html(fill_rates)

        # --- Create the combined output for Jupyter Notebook ---
        escaped_html = html.escape(fill_rate_table_html)
        notebook_display_html = f"""
        {fill_rate_table_html}
        <details style="margin-top: 20px; border: 1px solid #ccc; padding: 10px; border-radius: 5px;">
            <summary style="cursor: pointer; font-weight: bold; color: #0073e6;">Click to view/hide raw HTML code</summary>
            <pre style="background-color: #f8f8f8; padding: 10px; border: 1px solid #ddd; border-radius: 5px; white-space: pre-wrap; word-wrap: break-word;"><code>{escaped_html}</code></pre>
        </details>
        """

        # --- Display combined output in Jupyter Notebook ---
        print("\nDisplaying interactive fill rate table in notebook output:")
        display(HTML(notebook_display_html))

        # --- Save the clean HTML to a file ---
        with open(OUTPUT_HTML_PATH, 'w', encoding='utf-8') as f:
            f.write(fill_rate_table_html)

        print("\n" + "="*50)
        print("SUCCESS!")
        print(f"A clean version of the fill rate table has also been saved to '{OUTPUT_HTML_PATH}'")
        print("="*50 + "\n")

    except FileNotFoundError:
        print(f"Error: Input file not found at '{INPUT_CSV_PATH}'")
    except ImportError:
        print("\nNOTE: To display the table in the notebook, you must run this script in a Jupyter environment.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == '__main__':
    main()


Starting script...
Loaded 60824 rows and 517 columns.

Displaying interactive fill rate table in notebook output:


Variable,Fill Rate (%)
preop_creatinine,100.00%
last_preop_scr,100.00%
preop_scr,100.00%
hr,99.72%
preop_potassium,98.04%
preop_sodium,98.04%
preop_hct,97.96%
preop_hb,97.46%
preop_wbc,97.33%
preop_platelet,97.18%



SUCCESS!
A clean version of the fill rate table has also been saved to 'fill_rate_table.html'



# OLD CODE - DO NOT USE

In [1]:
import pandas as pd
output_csv = "/home/server/Projects/data/AKI/preop_data_test.csv"
df = pd.read_csv(output_csv)

In [2]:
import pandas as pd
import numpy as np


file = '/home/server/Projects/data/AKI/tabular_combined_unnormalized.csv'
df = pd.read_csv(file)

df_isna = df[df.filter(like='isna').columns]
col_fill_rate = 1 - (df_isna.sum(axis=0) / df_isna.shape[0])
col_fill_rate.index = col_fill_rate.index.str[:-5]

# cut down on all the repeat columns from taking eight metrics for each of these regular variables
high_frequency_labels = ["rr", "hr", "spo2", "fio2", "pmean", "etco2", "peep", 
"pip", "art_mbp", "cpat", "vt", "art_sbp", "art_dbp", 
"minvol", "pplat", "bt", "etgas", "cvp"]
medium_frequency_labels = ["pap_mbp", "pap_sbp", "pap_dbp", "nibp_mbp", "nibp_dbp", "nibp_sbp"]
regular_labels = high_frequency_labels + medium_frequency_labels
for label in regular_labels:
    cols = col_fill_rate.filter(like=label)
    value = min(cols)
    for col in cols.keys():
        col_fill_rate.pop(col)
    col_fill_rate[label] = value

# cleaning labels
col_fill_rate.index = col_fill_rate.index.str.replace(r"(mean_|sum_)", "", regex=True)

In [3]:
for i, fr in col_fill_rate.items():
    print(i, '\t', fr)

last_preop_scr 	 0.9999835591213995
min_preop_scr 	 0.9999835591213995
preop_total_protein 	 0.9616105484677101
preop_sodium 	 0.9803531500723399
preop_potassium 	 0.9804024727081415
preop_platelet 	 0.9717710114428515
preop_glucose 	 0.9414375904248323
preop_wbc 	 0.9733000131527029
preop_alt 	 0.9622681836117322
preop_chloride 	 0.9714093121136393
preop_lymphocyte 	 0.9248487439168749
preop_phosphorus 	 0.9513021175851637
preop_albumin 	 0.9651453373668288
preop_fibrinogen 	 0.844551492831777
preop_creatinine 	 1.0
preop_ptinr 	 0.958305931868999
preop_total_bilirubin 	 0.9329705379455479
preop_alp 	 0.960804945416283
preop_aptt 	 0.9545738524266737
preop_calcium 	 0.9661811127186637
preop_bun 	 0.9678580823359201
preop_ast 	 0.9625476785479417
preop_crp 	 0.7844765224253585
preop_hb 	 0.9746317243193476
preop_hct 	 0.9796297514139155
preop_seg 	 0.9246843351308693
air 	 0.7916447454951993
bis 	 0.6211528344074707
cbro2 	 0.0030415625411022162
ci 	 0.11462580560305147
dobui 	 0.02591

In [4]:
import pandas as pd
import numpy as np


file = '/home/server/Projects/data/AKI/tabular_combined_unnormalized.csv'
df = pd.read_csv(file)

# one row per subject, not operation
df = df.groupby('subject_id').agg('first').reset_index()
df = df.drop(columns=['op_id'])

S = ''
def sprint(s):
    global S
    S = S + s + '\n'

In [5]:
sprint(f'Total Population:\t{df.shape[0]}')
sprint('\nmetric\tmean, std')
#misc
mn_sd_cols = ['age', 'weight', 'height', 'BMI', 
              'BSA', 'asa', 'num_card_events', 
              'booking_case_length']
for col in mn_sd_cols:
    mn = df[col].mean()
    sd = df[col].std()
    sprint(f'{col}:\t{mn:.2f} ± {sd:.2f}')

sprint('\nmetric\tN\tpercent')

# sex
males = df.sex.value_counts()[True]
females = df.sex.value_counts()[False]
sprint(f'Male:\t{males}\t{males / df.shape[0] * 100:.2f}%')
sprint(f'Female:\t{females}\t{females / df.shape[0] * 100:.2f}%')

# asa
desc = df.asa.value_counts()
for key in desc.keys():
    value = desc[key]
    sprint(f'ASA {key}:\t{value}\t{value / df.shape[0] * 100:.3f}%')

# aki
desc = df.aki_boolean.value_counts()
value = desc[True]
sprint(f'AKI positivity:\t{value}\t{value / df.shape[0] * 100:.3f}%')

# departments
dept_cols = [col for col in df.columns if 'department' in col]
for col in dept_cols:
    value = df[col].sum()
    sprint(f'{col}:\t{value}\t{value / df.shape[0] * 100:.4f}%')

In [6]:
print(S)

Total Population:	49198

metric	mean, std
age:	58.05 ± 15.14
weight:	62.99 ± 11.72
height:	162.35 ± 8.99
BMI:	23.86 ± 3.64
BSA:	1.68 ± 0.19
asa:	1.84 ± 0.65
num_card_events:	0.88 ± 3.85
booking_case_length:	227.58 ± 122.04

metric	N	percent
Male:	25871	52.59%
Female:	23327	47.41%
ASA 2.0:	28730	58.397%
ASA 1.0:	14440	29.351%
ASA 3.0:	5628	11.439%
ASA 4.0:	373	0.758%
ASA 5.0:	27	0.055%
AKI positivity:	1726	3.508%
department_AN:	2	0.0041%
department_CTS:	6815	13.8522%
department_DM:	1	0.0020%
department_EM:	2	0.0041%
department_GS:	14380	29.2288%
department_IM:	23	0.0467%
department_NS:	7614	15.4762%
department_OG:	1407	2.8599%
department_OL:	1365	2.7745%
department_OS:	10964	22.2855%
department_OT:	380	0.7724%
department_PED:	0	0.0000%
department_PS:	885	1.7989%
department_RAD:	285	0.5793%
department_RO:	3	0.0061%
department_UR:	5072	10.3094%



In [7]:
df[df['department_PED'] == 1]

,subject_id,age,sex,height,weight,asa,emop,BSA,BMI,booking_case_length,...,sum_ffp_isna,sum_ftn_isna,sum_n2o_isna,sum_pc_isna,sum_pheresis_isna,sum_rbc_isna,sum_uo_isna,fluids_agg_isna,equiv_MAC_totals_isna,aki_boolean
